Most frequently used algorithm while training a neural network is back propagation. 
The model parameters are adjusted accroding to the gradient of the loss function wrt. a given parameter.

PyTorch has a built in differentiation engine called torch.autograd. It supports automatic computation of gradient for any computational graph.

In [2]:
import torch

In [14]:
x = torch.ones(5) # input tensor
y = torch.zeros(3) # expected output
w = torch.rand(5,3, requires_grad = True) # weight tensor
b = torch.rand(3, requires_grad = True) # bias tensor

z = torch.matmul(x, w) + b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z,y)


In [15]:
print(f"Gradient function for z = {z.grad}")
print(f"Gradient function for loss = {loss.grad_fn}")

Gradient function for z = None
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x00000219AB82F820>


C:\Users\kaise\AppData\Local\Temp\ipykernel_32736\1009803839.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\build\aten\src\ATen/core/TensorBody.h:493.)
  print(f"Gradient function for z = {z.grad}")


To optimize weights of the paramenters in a neural network, we need to compute the derivatives of our loss function wrt its parameters,
namely d(loss)/dw and d(loss)/db, under fixed values of x and y.

Using loss.backward() to calculate the derivatives. Then we can retreive the values from w.grad and b.grad

In [ ]:

loss.backward(retain_graph = True)
print(w.grad)
print(b.grad)

tensor([[0.3279, 0.3102, 0.3195],
        [0.3279, 0.3102, 0.3195],
        [0.3279, 0.3102, 0.3195],
        [0.3279, 0.3102, 0.3195],
        [0.3279, 0.3102, 0.3195]])
tensor([0.3279, 0.3102, 0.3195])


In [20]:
print(z, loss.item())

tensor([4.0978, 2.5953, 3.1374], grad_fn=<AddBackward0>) 3.3204689025878906


In [ ]:
z  = torch.matmul(x, w) + b
print(z.requires_grad)

with torch.no_grad():
    z = x.T @ w + b 
    # for @ first operand if it is a 1d tensor, visually treated as a column vector, and when trasnposed it become a row vector, but that is not the case for @ second operand, it is treated as a matrix, and when transposed it become a 2d tensor.
print(z.requires_grad)

# or we can use z.detach()
z = z.detach()
print(z.requires_grad)


True
False
False


In [28]:
k  = x.T @ w
x.T.shape

torch.Size([5])

In [23]:
w.shape

torch.Size([5, 3])

Directed Acyclic graphs
root --> Loss function,
leaves -> inpute tesnors

Tracing the graph from the root to the leaves, we can compute the the gradients using the chain rule.

In the forward pass, Autograd does 2 things:
1. run the requested operation to compute a resulting tensor
2. maintaing the operation's gradient function in the DAG

IN Backward pass, when .backward() is called on the root(loss), then autograd:
1. computes gradients from each .grad_fn
2. accumulated them in the respective tensor's .grad attribute and propagates all the way to the leaf tensors

Common Error : 

Notice that when we call backward for the second time with the same argument, the value of the gradient is different. 
This happens because when doing backward propagation, PyTorch accumulates the gradients, 
i.e. the value of computed gradients is added to the grad property of all leaf nodes of computational graph. 

If you want to compute the proper gradients, you need to zero out the grad property before. 

In real-life training an optimizer helps us to do this.


In [29]:
inp = torch.eye(4, 5, requires_grad=True)
out = (inp+1).pow(2).t()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"First call\n{inp.grad}")
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nSecond call\n{inp.grad}")
inp.grad.zero_()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nCall after zeroing gradients\n{inp.grad}")

First call
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])

Second call
tensor([[8., 4., 4., 4., 4.],
        [4., 8., 4., 4., 4.],
        [4., 4., 8., 4., 4.],
        [4., 4., 4., 8., 4.]])

Call after zeroing gradients
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])
